# 05 — *Optional*: unfreeze `layer4`

**Only run this if gate G4 came in short of 0.75.** ~15 hours on CPU, against about two
minutes for the frozen head, and it is the last item on the ablation list for exactly that
reason.

**Do not start it before G4 exists.** Without the frozen-body number there is nothing to say
whether the fifteen hours bought anything, and "it seemed better" is not a result.

If G4 came in *well* short — below ~0.60 — this is the wrong lever. At that level the usual
cause is a label or alignment fault rather than model capacity, and the fix is gate G2's
eyeball panel, not gradient descent.

Everything else is unchanged: same tiles, same splits, same class weights, same report. The
augmentation is switched on here because the body is now learning, and stain jitter is what
buys the stain tolerance that lets the pipeline skip normalisation entirely.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import backend_path            # noqa: F401 - puts the demo backend on sys.path
backend_path.ensure_dirs()
print("backend :", backend_path.BACKEND_ROOT)
print("tiles   :", backend_path.TILES_DIR)
print("models  :", backend_path.MODELS_DIR)

In [ ]:
import json

import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

import bcss, datasets, models, report

INIT = "imagenet"
INVERT_POLARITY = False
EPOCHS = 10
LR = 1e-4          # low: the body has already learnt edges and textures, and a large
                   # step would unlearn them before the head has anything useful to say
BATCH = 32
SEED = 0

torch.manual_seed(SEED)
rows = datasets.read_manifest(backend_path.TILES_DIR / "tiles_manifest.csv")
train_rows, test_rows = datasets.institution_split(rows)
datasets.assert_no_leak(train_rows, test_rows)

weights = datasets.class_weights(train_rows)
print(f"{len(train_rows):,} train / {len(test_rows):,} test tiles")
print(f"class weights: {dict(zip(bcss.CLASS_NAMES, [round(float(w), 3) for w in weights]))}")

In [ ]:
net = models.resnet18_backbone(INIT, weights_dir=backend_path.PRETRAINED_DIR)
models.unfreeze_layer4(net)

trainable = sum(p.numel() for p in net.parameters() if p.requires_grad)
total = sum(p.numel() for p in net.parameters())
print(f"{trainable:,} of {total:,} parameters trainable ({trainable / total:.0%}) - "
      "layer4 and the head")

# Augmentation ON: the body is learning now, so the stain jitter and the eight square
# symmetries are doing real work rather than perturbing a fixed feature extractor.
train_set = datasets.TileDataset(rows=train_rows, tiles_dir=backend_path.TILES_DIR,
                                 augment=True, invert=INVERT_POLARITY, seed=SEED)
test_set = datasets.TileDataset(rows=test_rows, tiles_dir=backend_path.TILES_DIR,
                                augment=False, invert=INVERT_POLARITY)

train_loader = DataLoader(train_set, batch_size=BATCH, shuffle=True, num_workers=0)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False, num_workers=0)

optimiser = torch.optim.AdamW(
    [p for p in net.parameters() if p.requires_grad], lr=LR, weight_decay=1e-4
)
loss_fn = torch.nn.CrossEntropyLoss(weight=weights)

## The run

Roughly 90 minutes per epoch on eight cores. Print the held-out number each epoch so a run
that is going nowhere can be stopped rather than waited out.

In [ ]:
history = []

for epoch in range(EPOCHS):
    net.train()
    running = 0.0
    for batch, batch_labels in tqdm(train_loader, desc=f"epoch {epoch}"):
        optimiser.zero_grad()
        loss = loss_fn(net(batch), batch_labels)
        loss.backward()
        optimiser.step()
        running += float(loss.item()) * len(batch)

    net.eval()
    predictions, truths = [], []
    with torch.inference_mode():
        for batch, batch_labels in test_loader:
            predictions.append(net(batch).argmax(dim=1).numpy())
            truths.append(batch_labels.numpy())

    scored = report.evaluate(np.concatenate(truths), np.concatenate(predictions))
    history.append({"epoch": epoch, "loss": running / len(train_set),
                    "dice_invasive": scored.dice_invasive, "accuracy": scored.accuracy})
    print(f"  epoch {epoch}: loss {history[-1]['loss']:.4f} | "
          f"dice invasive {scored.dice_invasive:.3f} | accuracy {scored.accuracy:.3f}")

    torch.save(net.state_dict(), backend_path.REPORTS_DIR / f"05_finetune_epoch{epoch}.pt")

## Did it earn the fifteen hours?

Compared against notebook 04's frozen-body number, and against the fold-to-fold spread. An
improvement smaller than that spread is not an improvement.

In [ ]:
frozen = json.loads(
    (backend_path.REPORTS_DIR / f"04_report_{INIT}.json").read_text()
)
frozen_dice = frozen["held_out"]["dice_invasive_vs_non_invasive"]
fold_sd = frozen["groupkfold_dice"]["sd"]
best = max(history, key=lambda h: h["dice_invasive"])

print(f"frozen body   : {frozen_dice:.3f}")
print(f"fine-tuned    : {best['dice_invasive']:.3f}  (epoch {best['epoch']})")
print(f"difference    : {best['dice_invasive'] - frozen_dice:+.3f}")
print(f"fold-to-fold sd of the frozen fit: {fold_sd:.3f}")

if best["dice_invasive"] - frozen_dice > 2 * fold_sd:
    print("\nG5 OK - the gain is larger than the fold noise. Publish this one.")
else:
    print("\nThe gain is inside the fold noise. Publish the frozen head instead:")
    print("it is two minutes to refit, it cannot overfit layer4 to 108 slides, and")
    print("this comparison is the evidence for choosing it.")

(backend_path.REPORTS_DIR / f"05_finetune_{INIT}.json").write_text(
    json.dumps({"history": history, "frozen_dice": frozen_dice, "fold_sd": fold_sd},
               indent=2), encoding="utf-8")